# CTIG — Cultural Text-to-Image Generation (Vietnam)

**Settings:** Accelerator = GPU T4 x2 (hoặc T4), Internet = On.

Lưu ý Kaggle: mỗi dòng `!lệnh` là một shell riêng, `!cd` không có tác dụng sang dòng sau. Dùng `%cd` hoặc `cd ... &&` trong cùng một dòng.

In [ ]:
REPO_URL = "https://github.com/OxyzGiaHuy/CTIG.git"
CONFIG   = "configs/kaggle_t4x2.yaml"      # 1 GPU: configs/kaggle_t4.yaml
REPO     = "/kaggle/working/CTIG"

import os, subprocess
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
%cd $REPO
!git pull -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!cd $REPO && pip install -q -r requirements.txt && pip install -q -e . --no-deps
import transformers, diffusers; print("transformers", transformers.__version__, "| diffusers", diffusers.__version__)

## (Tuỳ chọn) Khôi phục `runs` từ lần chạy trước
Nếu lần trước đã tải `runs_cache.zip` và upload lại thành một **Dataset** rồi *Add Input* vào notebook này, cell dưới sẽ giải nén để dùng lại cache stage 1–3, bằng chứng đã rút và ảnh tham chiếu. Không có thì cell tự bỏ qua.

In [ ]:
import glob, zipfile, os
for z in glob.glob("/kaggle/input/**/runs_cache.zip", recursive=True) + glob.glob("/kaggle/input/**/runs_full.zip", recursive=True):
    print("khôi phục từ", z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall("/kaggle/working")
print("runs hiện có:", os.listdir("/kaggle/working/runs") if os.path.exists("/kaggle/working/runs") else "(chưa có)")

## Smoke test: 2 prompt
Lần đầu tải model ~10 phút.

In [ ]:
!cd $REPO && python -m ctig.cli batch --config $CONFIG --ids p001,p050 --run-name smoke

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image, Markdown

def show(run_name, max_prompts=10):
    run = Path("/kaggle/working/runs") / run_name
    for r in json.loads((run / "records.json").read_text(encoding="utf-8"))[:max_prompts]:
        display(Markdown(f"### {r['prompt_id']} — {r['prompt_text']}  \n"
                         f"đạt **{r['passed']}** · CLIP {r['clip_fidelity']:.2f} · judge {r['judge_score']:.2f} · {r['iterations']} vòng"))
        for i, p in enumerate(r["iteration_images"]):
            display(Markdown(f"vòng {i}")); display(Image(filename=p, width=384))
        if r["residual_findings"]:
            display(Markdown("còn sót: " + "; ".join(r["residual_findings"][:3])))

show("smoke")

## Chạy đủ 50 prompt
~3 giờ trên 2×T4, ~6 giờ trên 1×T4. Kết quả ghi sau **mỗi** prompt. Chạy lại cùng lệnh sẽ dùng cache stage 1–3.

In [ ]:
!cd $REPO && python -m ctig.cli batch --config $CONFIG --run-name v1-full

In [ ]:
print(json.dumps(json.load(open("/kaggle/working/runs/v1-full/summary.json")), indent=1, ensure_ascii=False)[:1500])
show("v1-full", max_prompts=6)

## Đóng gói để tải về
Kaggle chỉ cho tải từng file, nên tạo các file đơn:

| File | Dùng để |
|---|---|
| `bundle.html` | mở bằng trình duyệt, ảnh đã nhúng |
| `bundle.json` | gửi cho người đánh giá pipeline |
| `runs_cache.zip` | **nhỏ**, chỉ cache stage 1–3 + bằng chứng + ảnh tham chiếu; upload thành Dataset để lần sau khôi phục |
| `runs_full.zip` | toàn bộ runs kể cả ảnh gốc, có thể vài trăm MB |

In [ ]:
RUN = "v1-full"   # hoặc "smoke"
!cd $REPO && python -m ctig.bundle /kaggle/working/runs/$RUN
!cd /kaggle/working && rm -f runs_cache.zip runs_full.zip && zip -qr runs_cache.zip runs/_cache && zip -qr runs_full.zip runs && ls -lh runs_cache.zip runs_full.zip
from IPython.display import FileLink, display
for f in (f"/kaggle/working/runs/{RUN}/bundle.html", f"/kaggle/working/runs/{RUN}/bundle.json",
          "/kaggle/working/runs_cache.zip", "/kaggle/working/runs_full.zip"):
    display(FileLink(f))

### Lần sau chạy lại
1. Tải `runs_cache.zip` về máy.
2. Kaggle → Datasets → New Dataset → upload file zip đó (giữ tên `runs_cache.zip`).
3. Trong notebook: Add Input → chọn dataset vừa tạo. Cell khôi phục ở trên sẽ tự giải nén, pipeline bỏ qua analysis/search/extract cho các prompt không đổi.

Hoặc bấm **Save Version** → *Quick Save* để Kaggle giữ `/kaggle/working` như output của version; tải zip toàn bộ từ tab Output.